[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VirtualFlyBrain/neurofly2026-workshop/blob/main/python/01_Discovery.ipynb)

# 01 · Discovery — finding neurons across datasets
**Problem P1:** *Find every instance of a neuron type across all datasets.*

VFB classifies neurons from every source with one ontology ([Drosophila Anatomy Ontology](https://www.ebi.ac.uk/ols4/ontologies/fbbt)), so a single query for a *type* returns individuals from FlyWire, hemibrain, BANC, male-CNS, MANC, optic-lobe and CATMAID at once — no wrangling nomenclature.

In [ ]:
# Run once per Colab session (skip if already installed locally).
# Belt-and-braces: build vfb_connect's two legacy sdist deps in an isolated
# modern environment first (immune to any broken setuptools in the runtime):
%pip install -q --use-pep517 jsonpath-rw colormath
%pip install -q vfb_connect navis flybrains neuprint-python python-catmaid plotly

## Route A — `vfb_connect`
Find the type name/symbol first via the [VFB search](https://virtualflybrain.org), then pull its instances. Substitute a type *you* care about.

**Worked example:** the DA1 lateral projection neuron (`DA1 lPN`, `FBbt_00067363`) — a well-studied olfactory PN. In VFB it resolves to **68 individuals** spanning hemibrain, FlyWire, BANC, male-CNS and FAFB. (It's *absent* from MANC and the optic lobe — it's a central-brain neuron, not in the nerve cord or optic lobe — a good reminder that not every cell is in every dataset.)

In [ ]:
# All individual neurons of a given type, across every dataset.
neuron_type = 'adult antennal lobe projection neuron DA1 lPN'   # DA1 lPN — change me
df = vfb.get_instances(neuron_type)   # returns a DataFrame
df.head(10)

In [ ]:
# Which datasets is this type represented in?
# Note: 'data_source' holds a *list* per row (a neuron can appear in more than one
# source), so explode it before counting.
col = 'data_source' if 'data_source' in df.columns else 'dataset'
df[col].explode().value_counts()


### Find neurons by location
The ontology also lets you ask 'what's *in* a region' — parts and overlapping cells.

In [ ]:
region_terms = vfb.get_terms_by_region('fan-shaped body')  # try 'medulla', 'mushroom body'
pd.DataFrame(region_terms).head()

### What's *new* since 2024?
The 2026-era connectomes already show up in the instance table above. Filter to the neurons that come from datasets added since the last workshop (BANC whole-CNS, male-CNS):

In [ ]:
# Which of these individuals come from datasets that did not exist at the 2024 workshop?
# 'data_source' is a list column, so test set-membership rather than string matching.
new_since_2024 = {'BANC', 'mc', 'ol'}   # whole-CNS, male-CNS, optic lobe
mask = df['data_source'].apply(lambda s: bool(new_since_2024 & set(s)))
new_sets = df[mask]
print(f'{len(new_sets)} of {len(df)} DA1 lPN individuals are in datasets new since 2024')
new_sets[['label', 'id', 'data_source']].head(10)


---
*This notebook is the Python route only. The MCP, chat, 3D-browser and R routes for this problem live on the [workshop page](https://workshop.virtualflybrain.org/problems/p1-discovery/).*

### 🧪 Your turn
Pick a neuron type from your own work. How many datasets is it in? Does **BANC** or **male-CNS** add instances that weren't available in 2024?